# Aegis — Evaluation Harness

This notebook runs automated evaluation for Aegis using:
- A Golden Dataset of missions
- Mocked agent execution (same as demo)
- Mocked LLM-as-Judge
- Metrics & plots for judge scores and regressions

It is self-contained and runs without API keys.

In [ ]:
!pip install rich numpy pandas matplotlib --quiet

In [ ]:
import json, time, uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rich.pretty import pprint

In [ ]:
# Attempt to load from Kaggle input, else use inline dataset
import os

kaggle_path = "/kaggle/input/golden-dataset/golden_dataset.json"
if os.path.exists(kaggle_path):
    with open(kaggle_path) as f:
        golden = json.load(f)
    print("Loaded golden dataset from Kaggle input.")
else:
    golden = [
        {"id": 1, "mission": "Create a landing page for a new smartwatch for students", "expected": "Landing page, copy, research"},
        {"id": 2, "mission": "Make a campaign for a fitness app in India", "expected": "Research + copy + deployment"},
        {"id": 3, "mission": "Summarize competitors for a new food delivery startup", "expected": "Competitor insights"},
        {"id": 4, "mission": "Plan a 2-day hackathon promotion targeted at university students", "expected": "Campaign plan, channels, copy"},
        {"id": 5, "mission": "Generate a content calendar for Product X for next month", "expected": "Calendar + topics"}
    ]
    print("Using inline golden dataset.")
pprint(golden[:3])

In [ ]:
# Minimal copy of demo system (trace, registry, agents, planner, judge)
TRACES = []
def clear_traces(): 
    global TRACES; TRACES = []

def trace_start(name): 
    span = {"id": str(uuid.uuid4()), "name": name, "start": time.time(), "events": []}
    TRACES.append(span)
    return span

def trace_end(span, result): 
    span["end"] = time.time(); span["duration"] = span["end"] - span["start"]; span["result"] = result

AGENTS = {}
def register_agent(name, func): AGENTS[name] = func
def call_agent(name, args):
    span = trace_start(f"call:{name}")
    res = AGENTS[name](args)
    trace_end(span, res)
    return res

# Agents
def market_agent(args): return {"insights": f"Research for '{args.get('query')}'", "score": 0.8}
def copy_agent(args): return {"copy": f"Generated copy for '{args.get('brief')}'"}
def deploy_agent(args): return {"url": "https://example.com/demo"}

register_agent("MarketResearch", market_agent)
register_agent("CopyWriter", copy_agent)
register_agent("WebDeployer", deploy_agent)

def planner(mission):
    return [("MarketResearch", {"query": mission}), ("CopyWriter", {"brief": mission}), ("WebDeployer", {"brief": mission})]

def run_mission(mission):
    clear_traces(); outputs = {}
    for name, args in planner(mission):
        outputs[name] = call_agent(name, args)
    return outputs, list(TRACES)  # return copy

In [ ]:
def judge_score(mission, trace):
    names = [s["name"] for s in trace]
    score = 0
    if any("MarketResearch" in n for n in names): score += 0.4
    if any("CopyWriter" in n for n in names): score += 0.3
    if any("WebDeployer" in n for n in names): score += 0.3
    return round(min(score, 1.0), 2)

In [ ]:
rows = []
for item in golden:
    mission = item["mission"]
    outputs, trace = run_mission(mission)
    score = judge_score(mission, trace)
    rows.append({"id": item.get("id"), "mission": mission, "score": score, "spans": len(trace)})

df = pd.DataFrame(rows)
df

In [ ]:
summary = {
    "mean_score": df['score'].mean(),
    "median_score": df['score'].median(),
    "min_score": df['score'].min(),
    "max_score": df['score'].max(),
    "missions": len(df)
}
pprint(summary)

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(df['id'], df['score'])
plt.xlabel("Mission ID")
plt.ylabel("Judge Score")
plt.title("Judge Scores per Mission")
plt.ylim(0,1.05)
plt.show()

In [ ]:
THRESHOLD = 0.75
if summary["mean_score"] < THRESHOLD:
    print("⚠️ Regression detected: mean score below threshold.")
else:
    print("✅ Mean score above threshold — no regression detected.")

# Save results for judges to inspect
df.to_csv("evaluation_results.csv", index=False)
print("Saved evaluation_results.csv")

## Notes for Judges

- This harness uses a mocked agent pipeline for reproducibility without API keys.
- Replace the mock functions with your deployed agents or LLM calls for full production evaluation.
- `evaluation_results.csv` contains per-mission judge scores and can be downloaded by reviewers.
- To reproduce full evaluation:
  - Run each mission end-to-end
  - Collect traces
  - Use an LLM-as-Judge prompt that evaluates trajectory + final output